# OP-C02: Python Pipeline Setup with SQLAlchemy

| What to expect | Value |
| --- | --- |
| Scenario | Retail sales operations |
| Difficulty | Easy |
| Delivery scope | Component exercise |
| Expected effort | 60–90 minutes |
| Deliverable | One Python module that owns configuration, database lifecycle, SQL execution, and report publication |
| Primary interface | Python with SQLAlchemy and pandas SQL I/O |
| Prerequisites | Python functions and basic SQL; OP-C01 recommended |
| Tooling | `python`, `sqlalchemy`, `pandas`, `postgresql`, `docker-compose` |
| Techniques | `environment-config`, `engine`, `connection`, `transaction`, `ddl`, `to-sql`, `read-sql`, `output-publication` |

OP-C01 built the runtime around the pipeline. OP-C02 zooms inside `pipeline.py` and practices everything except validation and transformation. Docker, Compose, Bash, the source data, and the pandas cleaning function are supplied.

## Start here

1. Complete only [`op_c02_sqlalchemy_pipeline_setup.py`](../../../src/big_data_example/labs/platform_operations/op_c02_sqlalchemy_pipeline_setup.py).
2. Do not edit the [supplied transform](../../../src/big_data_example/labs/platform_operations/op_c02_sales_transform.py) or [supplied runtime](../../../infra/interactive-data-engineering-labs/platform-operations/op-c02-sqlalchemy-pipeline-setup/README.md).
3. Follow the checkpoints in execution order and run the static checker before Docker.
4. After attempting the lab, open the [solution and explanation](../../../docs/interactive-data-engineering-labs/solutions/platform-operations/op-c02-sqlalchemy-pipeline-setup.md) (**spoiler**).


## Python setup map

```text
Compose environment variables
        ↓
read_runtime_config()
        ↓
build_report_query() + supplied build_valid_sales()
        ↓
create_database_engine(DATABASE_URL)
        ↓
with engine.begin() as connection
        ├── create schema and table
        ├── truncate and load DataFrame
        └── execute report query
        ↓ commit or rollback
engine.dispose() releases pooled connections
        ↓
write_report(output/sales_report.csv)
```

The sequence separates four different things that are easy to blur together: PostgreSQL is the server, the database is selected by the URL, the SQLAlchemy `Engine` manages connections, and a `Connection` executes work inside a transaction.


## Responsibility table

| Responsibility | Owner in this lab | Created when |
| --- | --- | --- |
| PostgreSQL process and initial `op_c02` database | Supplied Compose service and official PostgreSQL image | Container startup against an empty named volume |
| `INPUT_FILE`, `OUTPUT_FILE`, and `DATABASE_URL` variables | Supplied Compose pipeline service | Pipeline container startup |
| Python `Path` and URL values | Learner `read_runtime_config()` | Python process startup |
| SQLAlchemy connection manager | Learner `create_database_engine()` | Before database work |
| `reporting` schema and table | Learner DDL executed through a `Connection` | Inside the transaction |
| Table rows | Supplied DataFrame loaded by learner database code | Inside the transaction |
| Category report DataFrame | Learner SQL query | Inside the transaction |
| `sales_report.csv` | Learner output-publication function | After database work |


## API field guide: how to read the Python interfaces

An API tells you what operation is available, which arguments it accepts, what object it returns, what state it changes, and which errors it can raise. Read a call from left to right:

```text
owner.operation(required_argument, option=value) -> returned object
```

The owner matters. `pd.read_sql_query(...)` is a pandas module function that creates a DataFrame. `transactions_df.to_sql(...)` is a method on an existing DataFrame. `connection.execute(...)` is database work performed through one SQLAlchemy connection.

| API shape used by this lab | Returns | Side effect or failure to reason about |
| --- | --- | --- |
| `os.environ["NAME"]` | `str` | Raises `KeyError` when required configuration is missing. |
| `Path(text)` | `Path` | Constructs a path object; it does not create or validate a file. |
| `text(sql_string)` | `TextClause` | Wraps textual SQL; it does not execute the statement. |
| `create_engine(database_url)` | `Engine` | Configures the database dialect and connection pool; connection normally remains lazy. |
| `engine.begin()` | Context manager yielding a `Connection` | Commits on normal exit and rolls back when an exception escapes. |
| `connection.execute(statement)` | `CursorResult` | Sends the statement through that connection and participates in its transaction. |
| `transactions_df.to_sql(name, con=..., ...)` | Usually affected-row count or `None` | Writes DataFrame records and follows the supplied connection's transaction. |
| `pd.read_sql_query(sql, con=...)` | `DataFrame` | Executes a result-producing query and materializes its rows in memory. |
| `output_file.parent` | `Path` | Identifies the containing directory; it does not create it. |
| `path.mkdir(parents=..., exist_ok=...)` | `None` | Creates a directory; options control missing parents and an existing target. |
| `report_df.to_csv(path, ...)` | `None` when writing to a file | Creates or overwrites the output artifact. |
| `engine.dispose()` | `None` | Releases the engine's currently pooled connections; it does not stop PostgreSQL. |

Use the [Python environment mapping](https://docs.python.org/3/library/os.html#os.environ), [`pathlib` reference](https://docs.python.org/3/library/pathlib.html), [SQLAlchemy Engine creation API](https://docs.sqlalchemy.org/en/20/core/engines.html), and [Engine and Connection guide](https://docs.sqlalchemy.org/en/20/core/connections.html) when you need behavior beyond this table.


## API field guide: choose the options

The checkpoint contract tells you the desired behavior; the API options tell you how to express it. Do not copy every available option. Select the smallest set that protects the intended schema, transaction, and output.

### `DataFrame.to_sql()` decision table

Selected signature: `frame.to_sql(name, con, schema=None, if_exists="fail", index=True, method=None)`

| Parameter | Common choices | Decision question |
| --- | --- | --- |
| `name` | Table name string | What is the table name without the schema prefix? |
| `con` | SQLAlchemy `Engine` or `Connection` | Must this load share the surrounding transaction? |
| `schema` | Schema name or default schema | Which PostgreSQL namespace owns the table? |
| `if_exists` | `"fail"`, `"append"`, or `"replace"` | Should pandas preserve the explicitly created table contract? `replace` drops it. |
| `index` | `True` or `False` | Is the DataFrame index a real target column? |
| `method` | `None`, `"multi"`, or a callable | Should pandas group values into multi-row inserts, and does the database support that method? |

The complete [`DataFrame.to_sql()` reference](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.to_sql.html) includes additional batching and type options. The [`read_sql_query()` reference](https://pandas.pydata.org/docs/reference/api/pandas.read_sql_query.html) explains accepted query and connection objects.

### Output and SQL choices

| API or SQL clause | Choice this contract makes you reason about |
| --- | --- |
| `mkdir(parents=True, exist_ok=True)` | Create missing directory levels and allow a safe rerun when the directory already exists. |
| `to_csv(index=False, float_format=...)` | Exclude the pandas index and preserve the required revenue presentation. |
| `CREATE ... IF NOT EXISTS` | Make application object creation safe to repeat. |
| `TRUNCATE` followed by append | Implement a full refresh while retaining the explicit PostgreSQL table definition. |
| `GROUP BY` | Define one report row per category. |
| `SUM(...)` | Aggregate units and revenue within that grain. |
| `ORDER BY` | Make report row order deterministic. |

The [`to_csv()` API](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.to_csv.html) has many presentation and storage options. This lab needs only the options tied to its output contract.

### A repeatable API-reading routine

1. State the object you currently have and the object or side effect you need next.
2. Find the function or method owned by that object's library.
3. Identify required arguments before optional keyword arguments.
4. Read defaults carefully; defaults such as `index=True` or `if_exists="fail"` may not match the pipeline contract.
5. Check transaction ownership and destructive behavior before running the call.
6. Save the return value only when a later step needs it.


## Checkpoint 0: Verify the supplied environment

<details>
<summary>Open environment instructions and job connection</summary>

Run the next cell. It locates the repository, learner module, supplied transform, runtime, source CSV, and checker. It also verifies the Docker and Compose command-line tools without starting services.

**Acceptance evidence:** Every required path exists and both version commands exit successfully.

**Job connection:** Proving supplied dependencies first prevents setup mistakes from being misdiagnosed as database or Python defects.

</details>


In [ ]:
from pathlib import Path
import ast
import os
import subprocess
import sys

current_path = Path.cwd().resolve()
project_root = next(
    candidate
    for candidate in (current_path, *current_path.parents)
    if (candidate / "pyproject.toml").is_file()
    and (candidate / "AGENTS.md").is_file()
)
learner_file = project_root / "src/big_data_example/labs/platform_operations/op_c02_sqlalchemy_pipeline_setup.py"
transform_file = project_root / "src/big_data_example/labs/platform_operations/op_c02_sales_transform.py"
lab_directory = project_root / "infra/interactive-data-engineering-labs/platform-operations/op-c02-sqlalchemy-pipeline-setup"
source_csv = project_root / "data/samples/interactive-data-engineering-labs/platform-operations/op-c01-containerized-pipeline-runtime/sales.csv"
verify_script = project_root / "scripts/interactive-data-engineering-labs/platform-operations/op-c02-sqlalchemy-pipeline-setup/verify-lab.py"
lab_environment = os.environ.copy()
lab_environment["SALES_CSV_PATH"] = str(source_csv)

def run_command(command, *, cwd=lab_directory, environment=None):
    result = subprocess.run(
        [str(part) for part in command],
        cwd=cwd,
        env=environment,
        text=True,
        capture_output=True,
        check=False,
    )
    if result.stdout:
        print(result.stdout.rstrip())
    if result.stderr:
        print(result.stderr.rstrip())
    return result

def show_function(function_name):
    source = learner_file.read_text(encoding="utf-8")
    tree = ast.parse(source)
    node = next(
        item for item in tree.body
        if isinstance(item, (ast.FunctionDef, ast.AsyncFunctionDef)) and item.name == function_name
    )
    lines = source.splitlines()
    print("\n".join(lines[node.lineno - 1:node.end_lineno]))

for required_path in (learner_file, transform_file, lab_directory, source_csv, verify_script):
    assert required_path.exists(), f"Missing required path: {required_path}"

assert run_command(["docker", "--version"], cwd=project_root).returncode == 0
assert run_command(["docker", "compose", "version"], cwd=project_root).returncode == 0
print(f"Learner file: {learner_file}")


## Checkpoint 1: Read runtime configuration

<details>
<summary>Open checkpoint instructions, hint, and job connection</summary>

Target function: `read_runtime_config()`

Read `INPUT_FILE`, `OUTPUT_FILE`, and `DATABASE_URL` from `os.environ`. Convert the two file values to `Path` objects and return `(input_file, output_file, database_url)` in that order. Do not hard-code `/app` paths or credentials in Python.

**Acceptance evidence:** The completed function contains all three exact environment keys and returns two paths plus one string.

**Hint:** Compose owns the concrete runtime values; this function translates strings from the process environment into the Python objects needed by later stages.

**Job connection:** Twelve-factor-style configuration lets the same image run against different files and databases without rebuilding application code.

</details>


In [ ]:
show_function("read_runtime_config")


## Checkpoint 2: Build the report query

<details>
<summary>Open checkpoint instructions, SQL contract, and job connection</summary>

Target function: `build_report_query()`

Return a SQLAlchemy `text()` statement that selects `category`, `SUM(quantity) AS total_units`, and `SUM(sales_amount) AS total_revenue` from `reporting.sales_transactions`, groups by category, and orders by revenue descending then category ascending.

**Acceptance evidence:** The query is a `TextClause` and contains the required `GROUP BY`, both `SUM()` expressions, and deterministic `ORDER BY`.

**Hint:** `text()` does not execute SQL. It creates an executable SQLAlchemy statement that a later connection can run.

**Job connection:** Keeping a query object separate from connection and execution logic makes ownership and testing clearer.

</details>


In [ ]:
show_function("build_report_query")


## Checkpoint 3: Create the SQLAlchemy engine

<details>
<summary>Open checkpoint instructions, mental model, and job connection</summary>

Target function: `create_database_engine(database_url)`

Return `create_engine(database_url)`. Keep this function deliberately small.

The engine is a connection manager and pool configured by a URL. It does not start PostgreSQL, create the `op_c02` database, create the `reporting` schema, or necessarily open a connection at construction time.

**Acceptance evidence:** The function returns an `Engine` created from its argument rather than a hard-coded URL.

**Job connection:** Separating engine construction makes connection configuration replaceable for local, test, staging, and production environments.

</details>


In [ ]:
show_function("create_database_engine")


## Checkpoint 4: Create the schema and table

<details>
<summary>Open checkpoint instructions, DDL contract, and job connection</summary>

Target function: `create_schema_and_table(connection)`

Use `connection.execute(text(...))` twice. First create schema `reporting` if absent. Then create `reporting.sales_transactions` if absent with these columns: `transaction_id TEXT PRIMARY KEY`, `transaction_date DATE NOT NULL`, text customer/product/category columns, `quantity INTEGER NOT NULL`, `unit_price NUMERIC(12, 2) NOT NULL`, and `sales_amount NUMERIC(14, 2) NOT NULL`.

**Acceptance evidence:** Repeated execution leaves one schema and one table rather than failing because they already exist.

**Hint:** The official PostgreSQL image creates the database named by `POSTGRES_DB`; application DDL creates namespaces and tables inside that selected database.

**Job connection:** Infrastructure commonly owns server/database provisioning while an application or migration system owns its schema objects.

</details>


In [ ]:
show_function("create_schema_and_table")


## Checkpoint 5: Replace table rows from the supplied DataFrame

<details>
<summary>Open checkpoint instructions, object-shape note, and job connection</summary>

Target function: `replace_transactions(connection, transactions_df)`

Execute `TRUNCATE TABLE reporting.sales_transactions`, then call the DataFrame's `to_sql()` method with table name `sales_transactions`, schema `reporting`, the existing connection, `if_exists="append"`, `index=False`, and `method="multi"`.

Input object: a supplied eight-row × eight-column pandas DataFrame. Output: eight persisted rows in the existing PostgreSQL table. You are not cleaning or validating those rows in this lab.

**Acceptance evidence:** A rerun remains at eight rows instead of becoming sixteen.

**Hint:** `append` describes how `to_sql()` treats the existing table. `TRUNCATE` is what makes this particular pipeline a full refresh.

**Job connection:** Idempotent full-refresh behavior makes retries predictable; an incremental pipeline would need a different key, conflict, and state policy.

</details>


In [ ]:
show_function("replace_transactions")


## Checkpoint 6: Read and publish the report

<details>
<summary>Open checkpoint instructions, hints, and job connection</summary>

Targets: `read_report(connection, report_query)` and `write_report(report_df, output_file)`

Use `pd.read_sql_query(report_query, con=connection)` to return the database aggregation as a DataFrame. For publication, work on a copy, convert `total_revenue` with `pd.to_numeric(...).astype("float64")`, create the output parent directory, and write CSV with `index=False` and `float_format="%.2f"`.

**Acceptance evidence:** The report has two rows and three columns in SQL projection order, and its CSV is created at the configured output path with two decimal places.

**Hint:** The query runs before `engine.begin()` exits, so it sees the rows inserted in the same transaction. The file write happens afterward because it is not database state.

**Job connection:** Query execution and artifact publication cross different failure boundaries. Keeping them visible makes transaction and retry decisions easier to reason about.

</details>


In [ ]:
show_function("read_report")
print()
show_function("write_report")


## Checkpoint 7: Check my work — static setup contract

<details>
<summary>Open validation instructions and job connection</summary>

Run the next cell after completing all seven functions. It checks Python syntax and the required configuration, engine, DDL, load, query, and output operations. It also validates the supplied Compose and Bash runtime without starting containers.

**Acceptance evidence:** `PASS: OP-C02 Python setup satisfies the static contract`.

**Job connection:** Static checks catch missing lifecycle steps before slow builds or stateful database operations begin.

</details>


In [ ]:
static_check = run_command(
    [sys.executable, verify_script, "--mode", "static"],
    cwd=project_root,
    environment=lab_environment,
)
print(f"Static check exit code: {static_check.returncode}")


## Checkpoint 8: Run the supplied runtime

<details>
<summary>Open execution instructions and transaction explanation</summary>

Run the next cell only after the static check passes. The supplied script starts an isolated PostgreSQL service, builds your learner module into the pipeline image, and runs it once.

`main()` calls `engine.begin()`. Entering that context acquires a connection and begins a transaction. Normal exit commits schema, table, truncate, and insert work together. An exception rolls back the database transaction. The `finally` block disposes the engine's pooled connections even when work fails.

**Acceptance evidence:** The command exits `0`, loads eight rows, and reports the output file.

**Job connection:** Transaction scope is a correctness boundary. A pipeline should make it clear which operations commit together and which external effects, such as file publication, occur separately.

</details>


In [ ]:
pipeline_run = run_command(
    [str(lab_directory / "run_pipeline.sh")],
    environment=lab_environment,
)
assert pipeline_run.returncode == 0, "Pipeline failed; read the output above"


## Checkpoint 9: Check my work — runtime state

<details>
<summary>Open runtime validation and database-inspection instructions</summary>

The next cell verifies PostgreSQL health, eight persisted rows, and exact source-to-report totals. The following cell queries connection identity, schema/table state, and the report SQL directly through `psql`.

**Acceptance evidence:** The runtime checker reports `PASS`, `current_database` is `op_c02`, and the SQL report matches the generated CSV.

**Job connection:** Checking connection identity before trusting table results prevents a valid query against the wrong database from becoming false evidence.

</details>


In [ ]:
runtime_check = run_command(
    [sys.executable, verify_script, "--mode", "runtime"],
    cwd=project_root,
    environment=lab_environment,
)
assert runtime_check.returncode == 0, "Runtime check failed; read the output above"


In [ ]:
compose_prefix = ["docker", "compose", "--env-file", ".env", "-f", "docker-compose.yml"]
sql = """
SELECT current_database(), current_user;
SELECT COUNT(*) AS transaction_rows FROM reporting.sales_transactions;
SELECT category, SUM(quantity) AS total_units, SUM(sales_amount) AS total_revenue
FROM reporting.sales_transactions
GROUP BY category
ORDER BY total_revenue DESC, category ASC;
"""
run_command(
    [*compose_prefix, "exec", "-T", "postgres", "psql", "-U", "pipeline", "-d", "op_c02", "-c", sql],
    environment=lab_environment,
)
print((lab_directory / "output/sales_report.csv").read_text(encoding="utf-8"))


## Reflection

<details>
<summary>Open reflection questions</summary>

1. Which component creates the PostgreSQL process, database, schema, table, connection manager, and transaction?
2. Why is `text()` separate from `connection.execute()`?
3. Why should `to_sql()` receive the transaction's `connection` instead of opening unrelated database work?
4. What commits together inside `engine.begin()`, and what happens when an exception escapes the block?
5. Why does `engine.dispose()` appear in `finally`?
6. Why is the final CSV write outside the database transaction, and what production problem could that create?
7. For one completed function, identify the API owner, required arguments, selected options, return value, side effect, and failure behavior.
8. Which API defaults would have violated this lab's database or output contract if you had accepted them unchanged?

</details>


## Finish, restart, or reset

<details>
<summary>Open the state and reset guide</summary>

- Restarting or clearing the notebook does not change the learner Python file, containers, database volume, or report.
- Stop containers while retaining database data with `docker compose down` from the OP-C02 runtime directory.
- Remove only OP-C02's generated runtime state with `./reset-lab.sh --force`.
- Restore the learner module after the baseline is committed by closing its editor tab, inspecting the diff, and running:

```bash
git restore -- src/big_data_example/labs/platform_operations/op_c02_sqlalchemy_pipeline_setup.py
```

The reset script preserves the learner module and never targets OP-C01, the weekend project, or repository PostgreSQL databases.

</details>


## Definition of done

- Every learner function is implemented without changing the supplied transform or runtime.
- The static and runtime **Check my work** cells pass.
- The table contains eight rows after both the first and second runs.
- The SQL report and `output/sales_report.csv` agree.
- You can explain configuration, `text()`, Engine, Connection, transaction, DDL, `to_sql()`, `read_sql_query()`, output publication, and cleanup as separate responsibilities in execution order.
- You can read a new Python or pandas API signature, identify its owner, required inputs, defaults, returned object, side effects, and transaction implications before using it.
